# 03 · Measure overlap and remove duplicate predictions
**Level:** beginner → intermediate · **Time:** 40–50 minutes · **Runs offline on CPU**

**You will:** calculate intersection over union (IoU), implement greedy non-maximum suppression (NMS), compare class-aware and class-agnostic NMS, and check edge cases.

**Prerequisites:** notebook 01, NumPy arrays and simple loops. Predictions in this notebook are deliberately hand-crafted so you can inspect every decision.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw


## Step 1 — Compute intersection over union
**IoU = intersection area / union area.** It measures overlap between two boxes, from 0 (no overlap) to 1 (identical non-empty boxes). It does not know the object class or prediction confidence.

We use half-open/continuous `xyxy` coordinates, so area is `(x2-x1)*(y2-y1)` without `+1`. The implementation below returns zero for zero-area boxes and rejects inverted or non-finite coordinates.


In [ ]:
def valid_boxes(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return np.empty((0, 4), dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 4:
        raise ValueError("Expected boxes with shape (N, 4).")
    if not np.isfinite(values).all() or np.any(values[:, 2:] < values[:, :2]):
        raise ValueError("Boxes must be finite, with x2 >= x1 and y2 >= y1.")
    return values

def pairwise_iou(boxes_a, boxes_b):
    a, b = valid_boxes(boxes_a), valid_boxes(boxes_b)
    top_left = np.maximum(a[:, None, :2], b[None, :, :2])
    bottom_right = np.minimum(a[:, None, 2:], b[None, :, 2:])
    intersection = np.clip(bottom_right - top_left, 0, None).prod(axis=2)
    area_a = (a[:, 2:] - a[:, :2]).prod(axis=1)
    area_b = (b[:, 2:] - b[:, :2]).prod(axis=1)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

box_a = [[30, 30, 130, 130]]
box_b = [[80, 60, 180, 160]]
print("Intersection = 50 × 70 = 3500")
print("Union = 10000 + 10000 - 3500 = 16500")
print("IoU:", pairwise_iou(box_a, box_b)[0, 0])
canvas = Image.new("RGB", (230, 200), "white")
show_boxes(canvas, box_a + box_b, ["A", "B"], title="Overlapping boxes")
plt.show()


## Step 2 — Build trust with edge cases
Adjacent boxes that only share a boundary have zero intersection area. Empty input should produce an empty matrix, not a crash. Degenerate annotations have no useful area; training datasets should normally reject them even though this geometry function handles them.


In [ ]:
assert np.isclose(pairwise_iou([[0, 0, 10, 10]], [[0, 0, 10, 10]])[0, 0], 1)
assert pairwise_iou([[0, 0, 10, 10]], [[10, 0, 20, 10]])[0, 0] == 0
assert pairwise_iou([[0, 0, 0, 10]], [[0, 0, 0, 10]])[0, 0] == 0
assert pairwise_iou([], [[0, 0, 10, 10]]).shape == (0, 1)
try:
    pairwise_iou([[10, 0, 5, 10]], box_a)
except ValueError as error:
    print("Inverted coordinates caught:", error)
print("Geometry checks passed.")


## Step 3 — Inspect duplicate predictions
Imagine a detector has proposed several boxes around the same object. Each has a class and score. Filtering low scores removes weak candidates; NMS then removes overlapping duplicates. These are separate thresholds with separate purposes.

Here the rectangle and circle genuinely overlap. That makes the difference between class-aware and class-agnostic suppression visible.


In [ ]:
scene = Image.new("RGB", (340, 240), (240, 240, 240))
painter = ImageDraw.Draw(scene)
painter.rectangle((40, 45, 149, 174), fill=(230, 120, 100))
painter.ellipse((65, 65, 164, 164), fill=(90, 155, 235))
painter.rectangle((235, 75, 304, 164), fill=(230, 120, 100))
candidate_boxes = np.array([
    [40, 45, 150, 175], [44, 48, 151, 176], [33, 40, 145, 170],
    [65, 65, 165, 165], [69, 68, 168, 168], [235, 75, 305, 165],
    [170, 15, 215, 50]], dtype=float)
candidate_scores = np.array([0.95, 0.86, 0.65, 0.92, 0.80, 0.88, 0.18])
candidate_classes = np.array([0, 0, 0, 1, 1, 0, 1])
class_names = ["rectangle", "circle"]
candidate_labels = [f"{i}: {class_names[c]}" for i, c in enumerate(candidate_classes)]
show_boxes(scene, candidate_boxes, candidate_labels, candidate_scores, title="Seven candidate predictions")
plt.show()


## Step 4 — Implement greedy NMS
1. Discard scores below the confidence threshold and boxes with zero area.
2. Select the remaining box with the highest score.
3. Suppress remaining boxes whose IoU with it exceeds the NMS threshold.
4. Repeat until no candidates remain.

**Class-aware NMS** only suppresses boxes of the same class. **Class-agnostic NMS** compares all classes. Neither can guarantee that overlapping objects of the same class survive. This implementation breaks score ties by original index, making the classroom result repeatable.


In [ ]:
def nms(boxes, scores, classes, iou_threshold=0.5, confidence_threshold=0.25,
        class_aware=True):
    boxes = valid_boxes(boxes)
    scores = np.asarray(scores, dtype=float)
    classes = np.asarray(classes)
    if scores.shape != (len(boxes),) or classes.shape != (len(boxes),):
        raise ValueError("Provide one score and one class per box.")
    if not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):
        raise ValueError("Scores must be finite values in [0, 1].")
    if not 0 <= iou_threshold <= 1 or not 0 <= confidence_threshold <= 1:
        raise ValueError("Thresholds must be in [0, 1].")
    positive_area = np.all(boxes[:, 2:] > boxes[:, :2], axis=1)
    candidates = np.flatnonzero((scores >= confidence_threshold) & positive_area)
    order = candidates[np.argsort(-scores[candidates], kind="stable")]
    keep = []
    while len(order):
        best = order[0]
        keep.append(best)
        remaining = order[1:]
        overlaps = pairwise_iou(boxes[best:best + 1], boxes[remaining])[0]
        comparable = classes[remaining] == classes[best] if class_aware else np.ones(len(remaining), dtype=bool)
        order = remaining[~((overlaps > iou_threshold) & comparable)]
    return np.asarray(keep, dtype=int)

aware = nms(candidate_boxes, candidate_scores, candidate_classes)
agnostic = nms(candidate_boxes, candidate_scores, candidate_classes, class_aware=False)
print("Class-aware kept indices:", aware)
print("Class-agnostic kept indices:", agnostic)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, kept, title in zip(axes, [aware, agnostic], ["Class-aware NMS", "Class-agnostic NMS"]):
    show_boxes(scene, candidate_boxes[kept], [candidate_labels[i] for i in kept],
               candidate_scores[kept], ax=ax, title=title)
plt.tight_layout()
plt.show()


**Observe:** class-aware NMS can keep both overlapping classes. A lower **NMS IoU threshold** suppresses more aggressively. A higher **confidence threshold** removes more low-score predictions. Neither threshold is the IoU threshold used to declare a prediction correct during evaluation.

## Step 5 — Check NMS edge cases
The same class and identical geometry should collapse to one prediction. Different classes survive in class-aware mode. Our stable ordering means the first box wins an exact score tie.


In [ ]:
identical = [[0, 0, 10, 10], [0, 0, 10, 10]]
assert nms([], [], []).size == 0
assert nms(identical, [0.9, 0.9], [0, 0]).tolist() == [0]
assert nms(identical, [0.9, 0.8], [0, 1]).tolist() == [0, 1]
assert nms(identical, [0.9, 0.8], [0, 1], class_aware=False).tolist() == [0]
assert nms([[0, 0, 0, 10]], [0.9], [0]).size == 0
print("NMS checks passed.")


## Try it yourself
1. Sweep NMS IoU thresholds 0.1, 0.5, 0.9, 1.0. Which candidate indices survive?
2. Keep the NMS threshold fixed at 0.5 and sweep confidence thresholds 0.1, 0.5, 0.9.
3. Give the overlapping circle the same class as the rectangle. Explain why a real object may disappear.


## Solutions — compare your predictions with the code
At an NMS threshold of 1.0 no candidate is suppressed by overlap, because our rule is strictly `IoU > threshold`. Confidence and area filtering still apply.


In [ ]:
for threshold in [0.1, 0.5, 0.9, 1.0]:
    print("NMS IoU", threshold, "->", nms(candidate_boxes, candidate_scores, candidate_classes,
                                         iou_threshold=threshold).tolist())
for threshold in [0.1, 0.5, 0.9]:
    print("Confidence", threshold, "->", nms(candidate_boxes, candidate_scores, candidate_classes,
                                            confidence_threshold=threshold).tolist())
all_same_class = np.zeros_like(candidate_classes)
print("All one class ->", nms(candidate_boxes, candidate_scores, all_same_class).tolist())


## Recap and next step
IoU is geometry; confidence is a model score; NMS is postprocessing. Keeping these concepts separate helps explain detector outputs and tune a demonstration without confusing training and evaluation.

**Next:** [04 · Pretrained YOLO detection](04_Pretrained_YOLO_Detection.ipynb).

**Reference:** [Torchvision NMS API](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.nms.html) describes the standard greedy overlap rule. Our small NumPy implementation adds explicit validation and class-aware filtering for teaching.
